# VietMedBridge — 04: Embedding → hybrid retrieval → reranker → submission

Chạy **Runtime > Change runtime type > GPU (T4 trở lên)**, rồi Run all.
Notebook dùng candidate đã freeze; không cần crawl/chia chunk lại.
Mặc định đọc bản team đã xử lý `stage-a-data-v3-laodong`: 864 tài liệu,
9.076 child chunks. Đây là baseline pretrained trên corpus pilot; chưa fine-tune.

Một notebook cho toàn bộ bước retrieval. Bootstrap tự clone/cài package và
dùng code lock riêng. Checkpoint embedding theo lô, query theo từng ID được
lưu Drive; khi runtime ngắt, chạy lại từ đầu với cùng cấu hình và RUN_NAME.
Không chạy hai runtime ghi cùng RUN_NAME.

File submission bao phủ đủ 1.200 query, nhưng chỉ tìm trên corpus pilot.
Kết quả không chứng minh coverage toàn corpus hay đạt relevance gate/F2.


## 1. Bootstrap — mount Drive, clone repo, cài thư viện


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT đã dùng ở notebook 00–03.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "retrieval_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
reference = CODE_REVISION or lock.get("git_commit") or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook,retrieval]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "retrieval_runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 2. Chọn candidate và cấu hình

Với dữ liệu team đã freeze, giữ mặc định bên dưới. Nếu dùng candidate khác,
sửa BUILD_RUN/CANDIDATE_NAME đúng manifest từ notebook 03 và chọn RUN_NAME mới.
Không cần đổi code lock của data pipeline. Khi chủ động nâng retrieval code,
đặt CODE_REVISION trong Bootstrap và chọn RUN_NAME mới.

MAX_NEW_EMBEDDING_PARTS/MAX_NEW_QUERIES=None chạy hết. Có thể đặt số nhỏ để
giới hạn phiên; nếu chưa hoàn tất, notebook dừng trước bước xuất submission.
Giữ part_size và model revision khi resume; có thể giảm batch_size nếu cần.


In [ ]:
import re
import time
import torch
from transformers import AutoTokenizer
from vietmedbridge.artifacts import read_json, digest_json
from vietmedbridge.dataset import parquet_path
from vietmedbridge.retrieval_data import load_catalog, load_queries

BUILD_RUN = "stage-a-data-v3-laodong"
CANDIDATE_NAME = "candidate-1cd220a4be956d5a.json"
RUN_NAME = "stage-a-retrieval-v1"
MAX_NEW_EMBEDDING_PARTS = None
MAX_NEW_QUERIES = None
for value in (BUILD_RUN, CANDIDATE_NAME, RUN_NAME):
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", value):
        raise ValueError("Tên run/candidate phải là một filename, không chứa đường dẫn.")
if not torch.cuda.is_available():
    raise RuntimeError("Chọn GPU trong Runtime > Change runtime type rồi chạy lại.")
CONFIG = read_json(CHECKOUT / "configs/retrieval_baseline.json")
BUILD_DIR = DATA_ROOT / "processed" / BUILD_RUN
RUN_DIR = DATA_ROOT / "retrieval" / RUN_NAME
candidate_path = BUILD_DIR / CANDIDATE_NAME
if not candidate_path.is_file():
    raise FileNotFoundError(f"Không thấy {candidate_path}. Chọn candidate thực tế từ notebook 03.")
preview = read_json(candidate_path)
tokenizer_spec = preview["golden"]["tokenizer"]
if tokenizer_spec["model_id"] != CONFIG["dense"]["model_id"] or tokenizer_spec["revision"] != CONFIG["dense"]["revision"]:
    raise ValueError("Dense tokenizer phải khớp revision của candidate; không tự đổi representation.")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_spec["model_id"],
    revision=tokenizer_spec["revision"], use_fast=True, trust_remote_code=False)
CATALOG = load_catalog(BUILD_DIR, CANDIDATE_NAME, tokenizer, **CONFIG["pilot_limits"])
SNAPSHOT = read_json(DATA_ROOT / "raw/snapshot.json")
if SNAPSHOT["files"]["links_corpus.parquet"]["sha256"] != CATALOG.build_config["official_links_sha256"]:
    raise ValueError("Candidate không thuộc official corpus snapshot hiện tại.")
QUERIES = load_queries(parquet_path(DATA_ROOT, "query.parquet"), expected_count=CONFIG["expected_queries"])
QUERY_UNITS = [{"id": q["id"], "text": q["query"]} for q in QUERIES]
print("GPU:", torch.cuda.get_device_name(0))
print(json.dumps({"candidate": CATALOG.identity, "documents": len(CATALOG.documents),
    "children": len(CATALOG.children), "unique_dense_inputs": len(CATALOG.units),
    "queries": len(QUERIES), "checkpoint": str(RUN_DIR)}, ensure_ascii=False, indent=2))


## 3. BGE-M3 embedding — cần GPU

Encode mỗi representation duy nhất một lần, giữ aliases tới mọi official ID.
CLS pooling và L2 normalization; không thêm instruction cho query. Input dense
giữ budget 512 token gồm special tokens theo dense builder của candidate.
Checkpoint mặc định 256 inputs/lô, checksum và marker chỉ ghi khi lô hoàn tất.
Model tự giảm batch khi CUDA OOM. Runtime mới chỉ encode lô còn thiếu.


In [ ]:
from vietmedbridge.embeddings import embed_units, embedding_matrix
from vietmedbridge.retrieval_models import TorchDenseEncoder

dense = TorchDenseEncoder(CONFIG["dense"])
started = time.monotonic()
try:
    CORPUS_EMBEDDINGS = embed_units(CATALOG.units, dense, RUN_DIR / "corpus_embeddings",
        work_dir=WORK_DIR, max_new_parts=MAX_NEW_EMBEDDING_PARTS, **CONFIG["embedding"])
    if CORPUS_EMBEDDINGS["state"] != "COMPLETE":
        raise RuntimeError("Embedding corpus còn thiếu: chạy lại cell này để resume rồi chạy tiếp.")
    QUERY_EMBEDDINGS = embed_units(QUERY_UNITS, dense, RUN_DIR / "query_embeddings",
        work_dir=WORK_DIR, max_new_parts=MAX_NEW_EMBEDDING_PARTS, **CONFIG["embedding"])
    if QUERY_EMBEDDINGS["state"] != "COMPLETE":
        raise RuntimeError("Embedding query còn thiếu: chạy lại cell này để resume rồi chạy tiếp.")
    atomic_json(RUN_DIR / "dense_runtime.json", {"git_commit": CODE_COMMIT,
        "model": dense.identity, "gpu": torch.cuda.get_device_name(0),
        "seconds_this_call": time.monotonic() - started, "oom_backoffs_this_call": dense.oom_backoffs})
finally:
    dense.close()
    del dense
CORPUS_VECTORS = embedding_matrix(RUN_DIR / "corpus_embeddings", CORPUS_EMBEDDINGS)
QUERY_VECTORS = embedding_matrix(RUN_DIR / "query_embeddings", QUERY_EMBEDDINGS)
print("Corpus vectors:", CORPUS_VECTORS.shape, "| query vectors:", QUERY_VECTORS.shape)


## 4. BM25 + FAISS exact cosine index — CPU

BM25 dùng Unicode words và CJK unigrams/bigrams. Dense search dùng IndexFlatIP
trên vectors đã chuẩn hóa; RRF gộp top-100 dense và top-100 sparse cho mỗi query.
Đây là index pilot có giới hạn RAM, chưa thay thế benchmark ANN cho 100k/full corpus.


In [ ]:
from vietmedbridge.retrieval import HybridIndex, RetrievalConfig, predict_queries

INDEX = HybridIndex(CATALOG, CORPUS_VECTORS, CORPUS_EMBEDDINGS,
                    RUN_DIR / "index", work_dir=WORK_DIR)
RETRIEVAL_CONFIG = RetrievalConfig(**CONFIG["retrieval"])
print("Indexed unique representations:", INDEX.dense.ntotal)
print("Index signature:", INDEX.manifest["signature"])


## 5. BGE reranker → parent context — cần GPU

Rerank tối đa 40 query/child pairs bằng pretrained `bge-reranker-v2-m3`.
Hai model được nạp lần lượt để tiết kiệm VRAM. Chọn top-10 documents và
tối đa 8 parent chunks, tối đa 2 chunks/document; không bắt số docs bằng số chunks.
Tham số này là baseline để đo, chưa được tune theo nhãn. Chunk output là
nguyên văn source slice; model chỉ cho điểm, không sinh văn bản.

Mỗi query có checkpoint riêng. Đổi query/model/selection policy cần RUN_NAME mới.


In [ ]:
from vietmedbridge.retrieval_models import TorchReranker

reranker = TorchReranker(CONFIG["reranker"])
started = time.monotonic()
RERANKER_IDENTITY = dict(reranker.identity)
try:
    RECORDS, PREDICTION_REPORT = predict_queries(INDEX, QUERIES, QUERY_VECTORS,
        reranker, RUN_DIR / "queries", query_embedding_manifest=QUERY_EMBEDDINGS,
        config=RETRIEVAL_CONFIG, batch_size=CONFIG["embedding"]["batch_size"],
        max_new_queries=MAX_NEW_QUERIES)
    atomic_json(RUN_DIR / "reranker_runtime.json", {"git_commit": CODE_COMMIT,
        "model": reranker.identity, "gpu": torch.cuda.get_device_name(0),
        "seconds_this_call": time.monotonic() - started, "oom_backoffs_this_call": reranker.oom_backoffs})
finally:
    reranker.close()
    del reranker
print(json.dumps(PREDICTION_REPORT, ensure_ascii=False, indent=2))
if PREDICTION_REPORT["state"] != "COMPLETE":
    raise RuntimeError("Query còn thiếu: chạy lại cell này để resume trước khi xuất submission.")


## 6. Kiểm source/ID/schema và xuất submission — CPU

Kiểm đủ 1.200 query chính thức, official doc IDs, provenance và exact source
text của mỗi chunk. `submission.zip` chỉ chứa một `results.json` ở root.
Evidence/model/data manifests nằm ngoài ZIP. Không tính F2 khi chưa có nhãn
và scorer BTC; đây là submission pilot để thử đường chạy trên hệ thống chấm.


In [ ]:
from vietmedbridge.submission import export_submission

EXPORT = export_submission(RECORDS, QUERIES, CATALOG, PREDICTION_REPORT,
    RUN_DIR / "submission", expected_count=CONFIG["expected_queries"], work_dir=WORK_DIR,
    evidence={"git_commit": CODE_COMMIT, "dense": CORPUS_EMBEDDINGS["encoder"],
              "reranker": RERANKER_IDENTITY, "index": INDEX.manifest,
              "retrieval_config": CONFIG["retrieval"]})
print(json.dumps(EXPORT, ensure_ascii=False, indent=2))
print("ZIP:", RUN_DIR / "submission/submission.zip")
print("Corpus pilot:", len(CATALOG.documents), "documents; chưa đại diện toàn corpus.")
# Xem vài query cùng parent output trước khi upload lên hệ thống BTC.
for q, record in zip(QUERIES[:3], RECORDS[:3]):
    p = record["prediction"]
    print("\nQuery", q["id"], q["query"])
    print("Documents:", p["relevant_docs"])
    print("First chunk:", p["relevant_chunks"][0]["chunk_text"][:500])


## 7. Tải ZIP

Chạy cell này để tải file và tự upload lên hệ thống BTC. Khi nhận score,
giữ cả submission_manifest.json để so sánh baseline với lần mở rộng 100k
hoặc fine-tune sau này. Score thấp có thể do coverage pilot rất nhỏ; chưa
đủ để kết luận chọn model sai.


In [ ]:
from google.colab import files
files.download(str(RUN_DIR / "submission/submission.zip"))
